<a href="https://colab.research.google.com/github/MC648472/NLP-Abstention-Project/blob/feature%2FA-data-M1-M3/Abstention_in_Small_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# When Should a Small Language Model Say “I Don’t Know”?

### CISC7021 Applied Natural Language Processing — Student A part
**Data pipeline · evaluation protocol · M1 / M3 baselines · two-stage compute-aware abstainer**

This notebook is the runnable demo for the presentation. It downloads SQuAD 2.0,
builds the fixed 1,000-question val / test splits, loads Qwen2.5-0.5B-Instruct,
and runs the three methods end-to-end:

| Method | Idea | Cost |
|---|---|---|
| **M1** | length-normalised token log-prob of one greedy answer | 1 pass |
| **M3** | 5-sample lexical consistency (largest cluster share) | 5 samples |
| **Two-stage (ours)** | M1 screens; only borderline questions pay for M3 | 1 + 5p samples |

**Protocol:** thresholds are tuned on **val only**; test is scored once. Coverage is
the share answered; selective accuracy is accuracy on answered; overall accuracy
counts correct answers + correct refusals. The always-abstain baseline is **50.0%**
because half of the test questions are unanswerable.

**Default run:** 20-question end-to-end demo (`RUN_QUICK_DEMO=True`).
For the full 1,000-question table, set `RUN_FULL_EXPERIMENT=True` at the bottom of
the last code cell. To show the already-computed report table without a GPU, set
both flags to `False` and run `show_cached_results()`.


## How to run (Colab)

1. `Runtime -> Change runtime type -> T4 GPU`.
2. Clone the repo if needed: `!git clone https://github.com/MC648472/NLP-Abstention-Project.git`
   then `%cd NLP-Abstention-Project`.
3. Run the cells in order. The notebook finds `abstention_eval.py` in the repo
   root or in `src/`; B's `slm_interface.py` is optional (a local fallback is
   included, so the notebook also runs standalone).
4. Default mode is a **20-question end-to-end demo** (`RUN_QUICK_DEMO=True`).
   For the full 1,000-question table set `RUN_FULL_EXPERIMENT=True` in the last
   code cell; it stores 8 samples and scores the first 5, then writes
   `results/notebook_main_table.csv`.
5. To show only the already-computed report table (no GPU), set both flags to
   `False` and run `show_cached_results()`.


In [2]:
# ============================================================================
# CISC7021 Applied NLP — Abstention in Small Language Models (Student A)
# Data / evaluation / M1 / M3 / two-stage abstainer
# Model: Qwen2.5-0.5B-Instruct  |  Data: SQuAD 2.0
# 1,000 questions per split: 500 answerable + 500 unanswerable, seed 42
# Run: Runtime -> T4 GPU -> Run all. Default = 20-question demo.
# Full table: set RUN_FULL_EXPERIMENT=True in the last code cell.
# ============================================================================
import hashlib
import json
import math
import os
import random
import re
import string
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_NAME = "Abstention in Small Language Models"
IN_COLAB = os.path.isdir("/content")

if IN_COLAB:
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped: {exc}")


def _find_repo_root():
    """Find the repo root from the notebook's cwd, or from /content/* on Colab."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "abstention_eval.py").exists() or (p / "src" / "abstention_eval.py").exists():
            return p
    for p in Path("/content").glob("*/abstention_eval.py"):
        return p.parent
    for p in Path("/content").glob("*/src/abstention_eval.py"):
        return p.parent.parent
    return here


REPO_ROOT = _find_repo_root()
for _p in [REPO_ROOT, REPO_ROOT / "src", REPO_ROOT / "b_methods", Path.cwd()]:
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))


def seed_for_qid(qid):
    """Stable per-question seed: the same qid gets the same samples in any subset."""
    return int(hashlib.md5(str(qid).encode("utf-8")).hexdigest()[:8], 16) % (2**31 - 1)


DATA_DIR = os.environ.get(
    "SQUAD_DIR", "/content/squad_data" if IN_COLAB else str(REPO_ROOT / "squad_data")
)
OUT_DIR = os.environ.get("ABSTENTION_OUT", str(REPO_ROOT / "results"))
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_DIR, "train-v2.0.json")
DEV_FILE = os.path.join(DATA_DIR, "dev-v2.0.json")

B_HANDOFF_DIR = os.environ.get(
    "B_HANDOFF_DIR",
    os.path.join("/content/drive/MyDrive", "abstention", "data", "b_handoff")
    if os.path.isdir("/content/drive/MyDrive") else str(REPO_ROOT / "data" / "handoff"),
)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 32
TEMPERATURE = 0.7
N_SAMPLES = 8                 # store 8 samples for M3 / Stage 2
M3_K = 5                      # score the first 5, matching the report
VAL_FRAC = 0.5
MAX_PER_BUCKET = 500
BATCH_SIZE = 8                # batch size for greedy M1
QUICK_N = 5                   # questions in the live demo (keep small on CPU)
RUN_QUICK_DEMO = True         # default: 5-question demo on Colab
RUN_FULL_EXPERIMENT = False   # set True for the 1,000-question table
# (the same two flags are repeated at the bottom of the last code cell; edit there)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | model: {MODEL_NAME} | seed: {SEED}")
print(f"Repo: {REPO_ROOT}")
print(f"Data dir: {os.path.abspath(DATA_DIR)} | output dir: {os.path.abspath(OUT_DIR)}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | model: Qwen/Qwen2.5-0.5B-Instruct | seed: 42
Repo: /content
Data dir: /content/squad_data | output dir: /content/results


## 1. Data: SQuAD 2.0

SQuAD 2.0 nests data as `articles -> paragraphs -> qas`. Every QA entry gets flattened into one
record with `qid`, `question`, `context`, `answers` and `is_impossible`. The downloader only runs
when a file is missing, so a pre-uploaded copy in `DATA_DIR` is used as-is.


In [3]:
SQUAD_FILES = {
    TRAIN_FILE: "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json",
    DEV_FILE: "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json",
}


def ensure_squad_files():
    """Download the two SQuAD 2.0 files if they are not on disk yet."""
    import urllib.request

    for path, url in SQUAD_FILES.items():
        if os.path.exists(path) and os.path.getsize(path) > 0:
            continue
        print(f"downloading {url}")
        urllib.request.urlretrieve(url, path)


def load_squad_records(path):
    """Flatten a SQuAD 2.0 json file into a list of records."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    records = []
    for article in raw["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                answers = [a["text"].strip() for a in qa.get("answers", [])
                           if a["text"].strip()]
                records.append({
                    "qid": qa["id"],
                    "context": context,
                    "question": qa["question"].strip(),
                    "answers": answers,
                    "is_impossible": bool(qa.get("is_impossible", False)),
                })
    return records


def dataset_stats(records, name):
    answerable = sum(not r["is_impossible"] for r in records)
    print(f"{name:10s} total={len(records):6d}  answerable={answerable:6d}  "
          f"unanswerable={len(records) - answerable:6d}")


try:
    ensure_squad_files()
except Exception as exc:  # offline and no local copy
    raise RuntimeError(
        f"SQuAD 2.0 files are missing in {os.path.abspath(DATA_DIR)} and could not be "
        "downloaded. Upload train-v2.0.json / dev-v2.0.json there and re-run this cell."
    ) from exc

train_records = load_squad_records(TRAIN_FILE)
dev_records = load_squad_records(DEV_FILE)
dataset_stats(train_records, "train")
dataset_stats(dev_records, "dev")


downloading https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json
downloading https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json
train      total=130319  answerable= 86821  unanswerable= 43498
dev        total= 11873  answerable=  5928  unanswerable=  5945


## 2. Answer normalisation and correctness

Official SQuAD normalisation (lowercase, drop articles / punctuation, collapse whitespace).
A prediction on an answerable question counts as correct when it matches a gold answer by EM;
F1 is kept as a secondary score. On an unanswerable question the correct behaviour is to abstain.

`clean_prediction` trims chatty outputs (first line, "Answer:" prefixes, quotes) before scoring,
which matters for a 0.5B model that sometimes echoes the instruction.


In [4]:
def normalize_answer(text):
    """Lowercase, remove articles, punctuation and extra whitespace."""
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    return " ".join(text.split())


def em_score(prediction, gold):
    return int(normalize_answer(prediction) == normalize_answer(gold))


def f1_score(prediction, gold):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def score_against_golds(prediction, golds):
    em = max((em_score(prediction, g) for g in golds), default=0)
    f1 = max((f1_score(prediction, g) for g in golds), default=0.0)
    return em, f1


def clean_prediction(text):
    """Trim instruction-following noise before grading."""
    text = str(text).strip()
    text = text.split("\n")[0].strip()
    text = re.sub(r"^(the\s+)?answer\s*(is)?\s*[:\-]\s*", "", text, flags=re.IGNORECASE)
    return text.strip(" \"'`.")


# Apostrophes and punctuation are removed by normalize_answer, so every pattern
# below is written in the same normalised space ("don't" -> "dont").
REFUSAL_PATTERNS = [
    "i dont know", "i do not know", "dont know", "do not know",
    "cannot answer", "can not answer", "cant answer",
    "could not answer", "couldnt answer", "unable to answer",
    "not enough information", "insufficient information", "no information",
    "cannot be determined", "cant be determined", "cannot determine",
    "not specified", "not stated", "not mentioned", "unanswerable",
    "no answer", "no relevant information", "i cannot find",
    "do not have enough", "there is no information",
]


def is_abstention(text):
    norm = normalize_answer(text)
    if not norm:
        return True
    return any(pattern in norm for pattern in REFUSAL_PATTERNS)


def forced_correct(record, prediction):
    """Would this answer be correct if the system decided to answer?"""
    if record["is_impossible"]:
        return False
    em, _ = score_against_golds(prediction, record["answers"])
    return em == 1


def judge_record(record, prediction, abstain):
    """Correctness of the final behaviour: answer correctly, or refuse an unanswerable."""
    if record["is_impossible"]:
        return abstain, 0, 0.0
    if abstain:
        return False, 0, 0.0
    em, f1 = score_against_golds(prediction, record["answers"])
    return em == 1, em, f1


## 3. Fixed splits

SQuAD 2.0 ships only `train` and `dev`.

- `train` is left for Part B's M4 LoRA fine-tuning.
- `dev` is split by answerability into `val` (all threshold tuning) and `test` (final numbers only).

Each bucket is capped independently at `MAX_PER_BUCKET` questions, and the exact splits are written
to `OUT_DIR` so every method and both teammates evaluate on the same questions.

The same cell also writes the four files B needs (`b_items_{val,test}.jsonl` and
`b_gold_{val,test}.jsonl`) to `B_HANDOFF_DIR` (`MyDrive/abstention/data/b_handoff/` on Colab, or
`./b_handoff` locally). Those are exactly the questions to feed to `slm_interface.run_batch`.


In [5]:
def stratified_split(records, val_frac, seed):
    """Split records into val / test while keeping the answerable ratio."""
    rng = random.Random(seed)
    answerable = [r for r in records if not r["is_impossible"]]
    unanswerable = [r for r in records if r["is_impossible"]]
    rng.shuffle(answerable)
    rng.shuffle(unanswerable)

    def cut(items):
        k = int(len(items) * val_frac)
        return items[:k], items[k:]

    a_val, a_test = cut(answerable)
    u_val, u_test = cut(unanswerable)
    return a_val + u_val, a_test + u_test


def cap_buckets(records, max_per_bucket, seed):
    """Cap the answerable and unanswerable buckets independently."""
    rng = random.Random(seed)
    answerable = [r for r in records if not r["is_impossible"]]
    unanswerable = [r for r in records if r["is_impossible"]]
    rng.shuffle(answerable)
    rng.shuffle(unanswerable)
    return answerable[:max_per_bucket] + unanswerable[:max_per_bucket]


val_records, test_records = stratified_split(dev_records, VAL_FRAC, SEED)
val_records = cap_buckets(val_records, MAX_PER_BUCKET, SEED)
test_records = cap_buckets(test_records, MAX_PER_BUCKET, SEED)
dataset_stats(val_records, "val")
dataset_stats(test_records, "test")

for name, split in [("val", val_records), ("test", test_records)]:
    with open(os.path.join(OUT_DIR, f"squad2_{name}.json"), "w", encoding="utf-8") as f:
        json.dump(split, f, ensure_ascii=False, indent=2)
print(f"splits written to {os.path.abspath(OUT_DIR)}")

# The four files B needs for slm_interface.run_batch: exact items + gold.
os.makedirs(B_HANDOFF_DIR, exist_ok=True)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

for name, split in [("val", val_records), ("test", test_records)]:
    items = [{"id": r["qid"], "question": r["question"], "context": r["context"]}
             for r in split]
    gold = [{"id": r["qid"], "question": r["question"], "answers": r["answers"],
             "is_impossible": r["is_impossible"]} for r in split]
    write_jsonl(os.path.join(B_HANDOFF_DIR, f"b_items_{name}.jsonl"), items)
    write_jsonl(os.path.join(B_HANDOFF_DIR, f"b_gold_{name}.jsonl"), gold)
print(f"B handoff files written to {os.path.abspath(B_HANDOFF_DIR)}")


val        total=  1000  answerable=   500  unanswerable=   500
test       total=  1000  answerable=   500  unanswerable=   500
splits written to /content/results
B handoff files written to /content/drive/MyDrive/abstention/data/b_handoff


## 4. Inference through B's unified interface

`slm_interface.py` (owned by student B) is the single inference path for the whole team. The
notebook wraps `SLM` in a thin adapter that keeps A's list-based `generate_batch` contract:

- **M1** uses `SLM.generate(question, context)` and takes `avg_logprob` as the confidence;
- **M3** uses `SLM.sample(question, n=8, context=...)` and keeps the first 3/5/8 answers;
- the `SYSTEM_PROMPT` is imported from `slm_interface.py` and never re-defined, so all methods
  (A's M1/M3, B's M2/M4) are evaluated under the same prompt.

If `slm_interface.py` is missing, `LocalGenerator` falls back to the same model and the same prompt
so the notebook still runs standalone; results from the two paths are not mixed in one table.
The sequence confidence for M1 is the length-normalised mean token log-probability of the generated
answer tokens; `prob = exp(confidence)` is its sequence-level probability in `[0, 1]`.


In [6]:
# ---------------------------------------------------------------------------
# Inference is owned by student B: `slm_interface.py` lives in MyDrive (the
# setup cell already added MyDrive/Colab Notebooks and MyDrive to sys.path).
# If the file is absent, a local fallback with the same prompt keeps the
# notebook runnable standalone.
# ---------------------------------------------------------------------------
try:
    from slm_interface import SLM, SYSTEM_PROMPT
    import slm_interface as _slm_module
    HAS_B_INTERFACE = True
    print(f"slm_interface loaded from {_slm_module.__file__}")
except ImportError:
    SLM = None
    HAS_B_INTERFACE = False
    # Must stay identical to slm_interface.SYSTEM_PROMPT.
    SYSTEM_PROMPT = "Answer the question with a short phrase only. Do not explain."


class BGenerator:
    """Adapter: B's SLM -> A's list-based generate_batch contract."""

    def __init__(self):
        self.slm = None

    def load(self):
        if not HAS_B_INTERFACE:
            raise RuntimeError(
                "slm_interface.py was not found next to the notebook. Upload B's "
                "file, or use LocalGenerator for a standalone run."
            )
        self.slm = SLM()
        print("using slm_interface.SLM (student B's unified interface)")

    def generate_batch(self, questions, contexts, n_samples=1, return_logprobs=False,
                       do_sample=False, temperature=TEMPERATURE, batch_size=BATCH_SIZE,
                       seeds=None):
        # B's interface is per question; keep A's list-based contract so the
        # M1 / M3 / two-stage code above does not change.
        answers, logprobs = [], []
        for index, (question, context) in enumerate(zip(questions, contexts)):
            if n_samples == 1 and not do_sample:
                result = self.slm.generate(question, context)
                answers.append([clean_prediction(result["answer"])])
                if return_logprobs:
                    logprobs.append([float(result["avg_logprob"])])
            else:
                result = self.slm.sample(
                    question, n=n_samples, temperature=temperature,
                    context=context,
                    seed=(seeds[index] if seeds is not None else SEED + index),
                )
                answers.append([clean_prediction(a) for a in result["answers"]])
                if return_logprobs:
                    logprobs.append([0.0] * len(result["answers"]))
        return (answers, logprobs) if return_logprobs else answers

    def generate(self, question, context, n_samples=1, return_logprobs=False,
                 do_sample=False, temperature=TEMPERATURE, seeds=None):
        result = self.generate_batch(
            [question], [context], n_samples=n_samples,
            return_logprobs=return_logprobs, do_sample=do_sample,
            temperature=temperature, seeds=seeds,
        )
        if return_logprobs:
            answer_lists, logprob_lists = result
            return answer_lists[0], logprob_lists[0][0]
        return result[0]


class LocalGenerator:
    """Fallback used only when slm_interface.py is missing.

    It mirrors B's prompt and sampling settings so standalone numbers stay
    comparable with the team runs (batch generation is A's speed-up).
    """

    def __init__(self, model_name=MODEL_NAME, device=DEVICE):
        self.model_name = model_name
        self.device = device
        self.tokenizer = None
        self.model = None

    def load(self):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name, torch_dtype=dtype
        ).to(self.device)
        self.model.eval()
        print(f"loaded {self.model_name} on {self.device} (local fallback)")

    def _prompt(self, question, context):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context: {context}\nQuestion: {question}"},
        ]
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def _mean_logprob(self, scores, token_ids, row):
        values = []
        for step, token_id in zip(scores, token_ids.tolist()):
            log_probs = torch.log_softmax(step[row], dim=-1)
            values.append(log_probs[token_id].item())
        return sum(values) / len(values) if values else 0.0

    @torch.no_grad()
    def generate_batch(self, questions, contexts, n_samples=1, return_logprobs=False,
                       do_sample=False, temperature=TEMPERATURE, batch_size=BATCH_SIZE,
                       seeds=None):
        all_answers, all_logprobs = [], []
        sample = do_sample or n_samples > 1
        if sample and seeds is not None:
            batch_size = 1  # one question per batch, so each qid keeps its own seed

        for start in range(0, len(questions), batch_size):
            qs = questions[start:start + batch_size]
            cs = contexts[start:start + batch_size]
            prompts = [self._prompt(q, c) for q, c in zip(qs, cs)]
            enc = self.tokenizer(prompts, return_tensors="pt", padding=True)
            enc = {k: v.to(self.device) for k, v in enc.items()}
            prompt_len = enc["input_ids"].shape[1]

            kwargs = dict(
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=sample,
                num_return_sequences=n_samples,
                pad_token_id=self.tokenizer.eos_token_id,
                output_scores=True,
                return_dict_in_generate=True,
            )
            if sample:
                kwargs.update(temperature=temperature, top_p=0.95, top_k=50)

            if sample and seeds is not None:
                torch.manual_seed(int(seeds[start]))
            out = self.model.generate(**enc, **kwargs)
            for b in range(len(qs)):
                answers, logprobs = [], []
                for r in range(n_samples):
                    row = b * n_samples + r
                    gen_ids = out.sequences[row, prompt_len:]
                    eos = (gen_ids == self.tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
                    if len(eos):
                        gen_ids = gen_ids[:eos[0]]
                    text = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
                    answers.append(clean_prediction(text))
                    if return_logprobs:
                        logprobs.append(self._mean_logprob(out.scores, gen_ids, row))
                all_answers.append(answers)
                if return_logprobs:
                    all_logprobs.append(logprobs)

        return (all_answers, all_logprobs) if return_logprobs else all_answers

    @torch.no_grad()
    def generate(self, question, context, n_samples=1, return_logprobs=False,
                 do_sample=False, temperature=TEMPERATURE, seeds=None):
        result = self.generate_batch(
            [question], [context], n_samples=n_samples,
            return_logprobs=return_logprobs, do_sample=do_sample,
            temperature=temperature, batch_size=1, seeds=seeds,
        )
        if return_logprobs:
            answer_lists, logprob_lists = result
            return answer_lists[0], logprob_lists[0][0]
        return result[0]


def make_generator():
    """Use B's unified interface when available, otherwise the local fallback."""
    return BGenerator() if HAS_B_INTERFACE else LocalGenerator()


gen = make_generator()


slm_interface loaded from /content/slm_interface.py


## 5. Metric suite and calibration

Every prediction record carries `qid`, gold fields, `answer`, `abstain`, `correct`, `confidence`
(higher = more confident, used for AUROC / AUPRC and thresholding), `prob` (confidence mapped to
`[0, 1]`, used for ECE / Brier) and `n_samples` / `latency_s` for cost.

For M1, `prob = exp(mean token log-prob)`; for M3, `prob = consistency`. Calibration metrics are
therefore computed on a real probability scale instead of clipping negative log-probabilities to 0.


In [7]:
# ============================================================================
# Metrics live in ONE place now: b_handoff/abstention_eval.py -- the same file
# B scores with. Two copies of the metric code is exactly how the AUROC label
# bug happened (old behaviour label -> 0.486 vs forced_correct -> 0.739).
# ============================================================================
import os
import sys

# Locate abstention_eval.py: the Colab handoff dir (B_HANDOFF_DIR), the notebook's
# own folder, or MyDrive. Fixes ImportError when the script lives in MyDrive but
# B_HANDOFF_DIR only holds b_items / b_gold.
_REPO = globals().get("REPO_ROOT", os.getcwd())
_CANDIDATE_DIRS = [os.path.join(str(_REPO), "src"), str(_REPO),
                   globals().get("B_HANDOFF_DIR"),
                   os.path.join(os.getcwd(), "src"), os.getcwd(),
                   "/content/drive/MyDrive/Colab Notebooks", "/content/drive/MyDrive"]
for _d in _CANDIDATE_DIRS:
    if _d and os.path.isdir(_d) and os.path.isfile(os.path.join(_d, "abstention_eval.py")):
        if _d not in sys.path:
            sys.path.insert(0, _d)
        break
else:
    raise ImportError("abstention_eval.py not found: put it next to the notebook, in "
                      "B_HANDOFF_DIR, or in MyDrive.")

from abstention_eval import (  # noqa: E402
    coverage, selective_accuracy, risk, correct_refusal_rate, over_refusal_rate,
    auroc_auprc, forced_correct, judge, evaluate, format_metrics,
    tune_at_coverage, apply_threshold, tune_by_accuracy, tune_two_stage_at_coverage,
    brier, ece_score,
)
expected_calibration_error = ece_score
brier_score = brier


def _as_eval_record(p):
    # Notebook records use `qid`; the shared scorer keys on `id` + `forced_correct`.
    q = dict(p)
    q["id"] = q.get("id") or q.get("qid")
    if "forced_correct" not in q:
        q["forced_correct"] = forced_correct(q, q.get("answer", ""))
    return q


def evaluate_predictions(preds, include_calibration=True):
    metrics = evaluate([_as_eval_record(p) for p in preds])
    if not include_calibration:
        metrics.pop("ece", None)
        metrics.pop("brier", None)
    return metrics


## 6. M1 — single-pass token log-probability confidence

One greedy pass per question. Confidence is the length-normalised mean log-probability of the
generated answer tokens; `prob = exp(confidence)`. Abstain when the score is below `tau_m1`
(tuned on val) **or** when the model explicitly refuses (a verbal "I don't know" with high
confidence should not be turned into an answer).

`tune_threshold` maximises the F1 between predicted abstentions and the gold signal
"unanswerable OR the answer would be wrong", computed on val only.


In [8]:
def _gold_should_abstain(pred):
    if pred["is_impossible"]:
        return True
    return not forced_correct(pred, pred["answer"])


def tune_threshold(preds_val, target_coverage=None):
    """Val-only threshold. Delegates to the shared selector so the notebook and
    B's scorer cannot drift apart.

    Default objective = overall behaviour accuracy (correct answers + correct
    refusals). The old refusal-F1 objective rewards refusing everything, which
    is what pushed M1 to 11% coverage and M3 to tau=1.0. Pass target_coverage
    (0.3 / 0.5) to pick a fixed operating point instead.

    Returns (tau, score) where score is the val behaviour accuracy, or the
    achieved val coverage when target_coverage is given.
    """
    recs = [_as_eval_record(p) for p in preds_val]
    if target_coverage is not None:
        return tune_at_coverage(recs, target_coverage)
    return tune_by_accuracy(recs)




def apply_m1_threshold(preds, tau):
    out = []
    for p in preds:
        p = dict(p)
        p["abstain"] = p.get("verbal_abstain", False) or p["confidence"] < tau
        p["correct"], p["em"], p["f1"] = judge_record(p, p["answer"], p["abstain"])
        out.append(p)
    return out


def run_m1(generator, records, batch_size=BATCH_SIZE):
    """One greedy pass per question; confidence = mean token log-probability."""
    questions = [r["question"] for r in records]
    contexts = [r["context"] for r in records]
    seeds = [seed_for_qid(r["qid"]) for r in records]
    start = time.time()
    answers, logprobs = generator.generate_batch(
        questions, contexts, n_samples=1, return_logprobs=True,
        do_sample=False, batch_size=batch_size, seeds=seeds,
    )
    per_question = (time.time() - start) / max(1, len(records))

    preds = []
    for i, record in enumerate(records):
        answer = answers[i][0]
        confidence = logprobs[i][0]
        verbal = is_abstention(answer)
        correct, em, f1 = judge_record(record, answer, verbal)
        preds.append({
            "qid": record["qid"],
            "question": record["question"],
            "context": record["context"],
            "answers": record["answers"],
            "is_impossible": record["is_impossible"],
            "answer": answer,
            "abstain": verbal,
            "verbal_abstain": verbal,
            "confidence": float(confidence),
            "prob": float(math.exp(confidence)),
            "correct": correct,
            "em": em,
            "f1": f1,
            "n_samples": 1,
            "latency_s": per_question,
        })
    return preds


## 7. M3 — multi-sample consistency

Sample `N_SAMPLES` answers at temperature 0.7, cluster them by normalised lexical match /
containment, and take the largest-cluster share as the confidence score. The representative answer
is the largest cluster's member.

A question whose majority answer is an explicit refusal is treated as an abstention regardless of
the consistency value; otherwise M3 would answer "I don't know" five times with consistency 1.0.
`tau_c` is tuned on val with the same refusal-F1 objective.


In [9]:
def cluster_answers(answers):
    """Group answers by normalised exact match / containment."""
    clusters = []  # list of (normalised representative, [raw answers])
    for raw in answers:
        norm = normalize_answer(raw)
        placed = False
        for representative, members in clusters:
            if norm == representative or (
                norm and representative and (norm in representative or representative in norm)
            ):
                members.append(raw)
                placed = True
                break
        if not placed:
            clusters.append((norm, [raw]))
    return [members for _, members in clusters]


def consistency_score(answers):
    if not answers:
        return 0.0, 0
    clusters = cluster_answers(answers)
    return max(len(c) for c in clusters) / len(answers), len(clusters)


def majority_answer(answers):
    clusters = cluster_answers(answers)
    return max(clusters, key=len)[0]


def run_m3(generator, records, n_samples=N_SAMPLES, batch_size=max(1, BATCH_SIZE // 2),
           seed_mode="qid"):
    questions = [r["question"] for r in records]
    contexts = [r["context"] for r in records]
    if seed_mode == "index":
        # B's run_batch used seed=0+i on the full items list; use the same
        # scheme here so a full rerun matches the cached M3 outputs.
        seeds = list(range(len(records)))
    else:
        seeds = [seed_for_qid(r["qid"]) for r in records]
    start = time.time()
    sampled = generator.generate_batch(
        questions, contexts, n_samples=n_samples,
        return_logprobs=False, do_sample=True, batch_size=batch_size,
        seeds=seeds,
    )
    per_question = (time.time() - start) / max(1, len(records))

    preds = []
    for record, answers in zip(records, sampled):
        consistency, n_clusters = consistency_score(answers)
        answer = majority_answer(answers)
        refusal = is_abstention(answer)
        correct, em, f1 = judge_record(record, answer, refusal)
        preds.append({
            "qid": record["qid"],
            "question": record["question"],
            "context": record["context"],
            "answers": record["answers"],
            "is_impossible": record["is_impossible"],
            "answer": answer,
            "sample_answers": answers,
            "n_clusters": n_clusters,
            "abstain": refusal,
            "verbal_abstain": refusal,
            "confidence": float(consistency),
            "prob": float(consistency),
            "correct": correct,
            "em": em,
            "f1": f1,
            "n_samples": n_samples,
            "latency_s": per_question,
        })
    return preds


def rescore_m3(preds, k):
    """Recompute M3 from the first k stored samples (no new generation)."""
    out = []
    for p in preds:
        answers = p["sample_answers"][:k]
        consistency, n_clusters = consistency_score(answers)
        answer = majority_answer(answers)
        refusal = is_abstention(answer)
        q = dict(p)
        q.update({
            "answer": answer,
            "abstain": refusal,
            "verbal_abstain": refusal,
            "confidence": float(consistency),
            "prob": float(consistency),
            "n_samples": k,
            "n_clusters": n_clusters,
        })
        q["correct"], q["em"], q["f1"] = judge_record(p, answer, refusal)
        out.append(q)
    return out


def apply_m3_threshold(preds, tau_c):
    out = []
    for p in preds:
        p = dict(p)
        p["abstain"] = p.get("verbal_abstain", False) or p["confidence"] < tau_c
        p["correct"], p["em"], p["f1"] = judge_record(p, p["answer"], p["abstain"])
        out.append(p)
    return out


## 8. Two-stage compute-aware abstainer

1. **Stage 1 (M1 screening).** One greedy pass gives `s1`.
   - `s1 >= tau_high`: answer directly.
   - `s1 <= tau_low`: abstain directly.
   - otherwise **borderline**.
2. **Stage 2 (M3 refinement).** Only borderline questions are sampled `N_SAMPLES` times; the
   consistency threshold `tau_c` decides answer vs abstain.

Cost is recorded per question: 1 sequence for high/low, `1 + N_SAMPLES` for borderline.

For tuning, `tune_two_stage` reuses **cached** M1 and M3 val predictions and simulates every
threshold pair, so the 5x5 grid costs one M1 pass and one M3 pass instead of 25 full pipelines.


In [10]:
def simulate_two_stage(m1_preds, m3_preds, tau_low, tau_high, tau_c):
    """Build two-stage predictions from cached M1 / M3 val outputs (no generation)."""
    m3_by_qid = {p["qid"]: p for p in m3_preds}
    preds = []
    for p1 in m1_preds:
        s1 = p1["confidence"]
        stage1_answer = p1["answer"]
        if s1 >= tau_high:
            stage = "high"
            answer = stage1_answer
            abstain = p1.get("verbal_abstain", False)
            used = 1
            consistency = None
            confidence = p1["prob"]
        elif s1 <= tau_low:
            stage = "low"
            answer = stage1_answer
            abstain = True
            used = 1
            consistency = None
            confidence = p1["prob"]
        else:
            stage = "borderline"
            p3 = m3_by_qid[p1["qid"]]
            answer = p3["answer"]
            consistency = p3["confidence"]
            abstain = p3.get("verbal_abstain", False) or consistency < tau_c
            used = 1 + p3["n_samples"]
            confidence = consistency

        correct, em, f1 = judge_record(p1, answer, abstain)
        preds.append({
            "qid": p1["qid"],
            "question": p1["question"],
            "context": p1["context"],
            "answers": p1["answers"],
            "is_impossible": p1["is_impossible"],
            "answer": answer,
            "stage1_score": s1,
            "stage": stage,
            "consistency": consistency,
            "abstain": abstain,
            "confidence": float(confidence),
            "prob": float(confidence),
            "correct": correct,
            "em": em,
            "f1": f1,
            "n_samples": used,
            "latency_s": (p1.get("latency_s") or 0.0)
            + ((m3_by_qid[p1["qid"]].get("latency_s") or 0.0)
               if stage == "borderline" else 0.0),
        })
    return preds


def tune_two_stage(m1_val, m3_val, target_coverage=0.5, quantiles=None):
    """Val-only: pick (tau_low, tau_high, tau_c) for a target coverage.

    Delegates to the shared scorer (the same code that writes the report table),
    so the notebook and run_fixed_coverage.py cannot drift. Objective = overall
    behaviour accuracy among configs whose val coverage is within 2 points of the
    target; ties go to fewer samples. The old
    `selective_accuracy - 0.5 * over_refusal_rate` objective is gone: it is not
    comparable across methods (selective accuracy already depends on coverage)
    and it ignores correctly refused questions.
    """
    gold_val = {r["qid"]: {"is_impossible": r["is_impossible"], "answers": r["answers"]}
                for r in val_records}
    cfg, acc, cov = tune_two_stage_at_coverage(
        [_as_eval_record(p) for p in m1_val],
        [_as_eval_record(p) for p in m3_val],
        gold_val, target_coverage, quantiles=quantiles,
    )
    return {"config": cfg, "metrics": None, "objective": float(acc), "coverage": float(cov)}


def run_two_stage(generator, records, tau_low, tau_high, tau_c,
                  n_samples=N_SAMPLES, batch_size=BATCH_SIZE):
    """Run the real two-stage pipeline on a record list."""
    start = time.time()
    m1_preds = run_m1(generator, records, batch_size=batch_size)
    borderline = [p for p in m1_preds if tau_low < p["confidence"] < tau_high]
    m3_preds = run_m3(generator, borderline, n_samples=n_samples) if borderline else []

    preds = []
    m3_by_qid = {p["qid"]: p for p in m3_preds}
    for p1 in m1_preds:
        s1 = p1["confidence"]
        if s1 >= tau_high:
            stage, answer = "high", p1["answer"]
            abstain = p1.get("verbal_abstain", False)
            used, consistency, confidence = 1, None, p1["prob"]
        elif s1 <= tau_low:
            stage, answer = "low", p1["answer"]
            abstain = True
            used, consistency, confidence = 1, None, p1["prob"]
        else:
            p3 = m3_by_qid[p1["qid"]]
            stage, answer = "borderline", p3["answer"]
            consistency = p3["confidence"]
            abstain = p3.get("verbal_abstain", False) or consistency < tau_c
            used, confidence = 1 + p3["n_samples"], consistency

        correct, em, f1 = judge_record(p1, answer, abstain)
        preds.append({
            "qid": p1["qid"],
            "question": p1["question"],
            "context": p1["context"],
            "answers": p1["answers"],
            "is_impossible": p1["is_impossible"],
            "answer": answer,
            "stage1_score": s1,
            "stage": stage,
            "consistency": consistency,
            "abstain": abstain,
            "confidence": float(confidence),
            "prob": float(confidence),
            "correct": correct,
            "em": em,
            "f1": f1,
            "n_samples": used,
            "latency_s": None,
        })

    per_question = (time.time() - start) / max(1, len(records))
    for p in preds:
        p["latency_s"] = per_question
    return preds


## 9. Curves and ablations

- `coverage_accuracy_curve` sweeps the confidence threshold and returns coverage / selective
  accuracy / risk points for the report figures.
- `sample_count_ablation` tunes `tau_c` on val for each sample count `{3, 5, 8}` and reports the
  test metrics together with the average number of sequences per question.


In [11]:
def coverage_accuracy_curve(preds, n_points=25):
    ordered = sorted(preds, key=lambda p: p["confidence"])
    curve = []
    for k in range(1, len(ordered) + 1):
        tau = ordered[k - 1]["confidence"]
        decided = []
        for p in ordered:
            q = dict(p)
            q["abstain"] = q.get("verbal_abstain", False) or q["confidence"] < tau
            q["correct"], _, _ = judge_record(q, q["answer"], q["abstain"])
            decided.append(q)
        metrics = evaluate_predictions(decided)
        curve.append({"threshold": float(tau), "coverage": metrics["coverage"],
                      "selective_accuracy": metrics["selective_accuracy"],
                      "risk": metrics["risk"]})
    step = max(1, len(curve) // n_points)
    return curve[::step]


def sample_count_ablation(generator, val_records, test_records, counts=(3, 5, 8)):
    """Sample the largest k once per split, then slice to 3/5/8 (B's advice)."""
    max_k = max(counts)
    val_full = run_m3(generator, val_records, n_samples=max_k)
    test_full = run_m3(generator, test_records, n_samples=max_k)
    rows = []
    for k in counts:
        val_preds = rescore_m3(val_full, k)
        tau_c, _ = tune_threshold(val_preds)
        test_preds = apply_m3_threshold(rescore_m3(test_full, k), tau_c)
        metrics = evaluate_predictions(test_preds)
        metrics["n_samples_setting"] = k
        metrics["tau_c"] = tau_c
        rows.append(metrics)
    return rows


def cost_summary(m1_test, m3_test, ts_test):
    rows = []
    for name, preds in [("M1", m1_test), ("M3", m3_test), ("Two-stage", ts_test)]:
        metrics = evaluate_predictions(preds)
        rows.append({
            "method": name,
            "avg_samples": metrics["avg_samples"],
            "avg_latency_s": metrics["avg_latency_s"],
            "coverage": metrics["coverage"],
            "selective_accuracy": metrics["selective_accuracy"],
        })
    return rows


## 10. Running the experiments

Two modes, both in the last code cell:

- **Quick demo (default)**: 20 val questions end-to-end. M1 screens, borderline
  questions go to 5-sample M3, and the result is printed as a small table. This
  is the mode for the live Colab demonstration (about 1-2 minutes on T4).
- **Full experiment**: 1,000 questions per split, 8 samples stored and the first
  5 scored, fixed coverage 0.3 and 0.5, thresholds tuned on val only. This writes
  `results/main_table_fixed_coverage.csv` in the same schema as the report.

The report numbers come from the **full** mode; the demo is only for showing the
pipeline. If you just want to display the already-computed table, set both flags
to `False` and run `show_cached_results()`.


In [12]:
import pandas as pd
from IPython.display import display


def preds_to_dataframe(preds):
    rows = []
    for p in preds:
        rows.append({
            "qid": p["qid"],
            "is_impossible": p["is_impossible"],
            "answer": p["answer"],
            "abstain": p["abstain"],
            "confidence": p["confidence"],
            "prob": p.get("prob"),
            "correct": p["correct"],
            "em": p.get("em"),
            "f1": p.get("f1"),
            "stage": p.get("stage"),
            "n_samples": p["n_samples"],
            "latency_s": p.get("latency_s"),
        })
    return pd.DataFrame(rows)


def split_error_cases(preds):
    buckets = {"correct_answer": [], "incorrect_answer": [],
               "unnecessary_refusal": [], "missed_refusal": []}
    for p in preds:
        if p["is_impossible"]:
            if p["abstain"]:
                continue
            buckets["missed_refusal"].append(p)
        elif p["abstain"]:
            buckets["unnecessary_refusal"].append(p)
        elif p["correct"]:
            buckets["correct_answer"].append(p)
        else:
            buckets["incorrect_answer"].append(p)
    return buckets


def save_run(method_name, preds, metrics, out_dir=OUT_DIR):
    df = preds_to_dataframe(preds)
    df.to_csv(os.path.join(out_dir, f"{method_name}_preds.csv"), index=False)
    with open(os.path.join(out_dir, f"{method_name}_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)
    return df


def _metric_row(method, target, tau, val_cov, metrics, note=""):
    return {
        "method": method,
        "target_cov": target,
        "val_tau": round(float(tau), 4) if tau is not None else "",
        "val_cov": round(float(val_cov), 4),
        "test_cov": round(float(metrics["coverage"]), 4),
        "selective_acc": round(float(metrics["selective_accuracy"]), 4),
        "overall_acc": round(float(metrics["behaviour_accuracy"]), 4),
        "correct_refusal": round(float(metrics["correct_refusal_rate"]), 4),
        "over_refusal": round(float(metrics["over_refusal_rate"]), 4),
        "avg_samples": round(float(metrics["avg_samples"]), 2),
        "note": note,
    }


def run_full_pipeline(generator, val_records, test_records, targets=(0.3, 0.5),
                      run_ablations=False):
    """Fixed-coverage evaluation: store 8 samples, score 5, tune on val only."""
    generator.load()

    print("M1: val ...")
    m1_val = run_m1(generator, val_records)
    print("M1: test ...")
    m1_test = run_m1(generator, test_records)

    print(f"M3: val ({N_SAMPLES} samples) ...")
    m3_val_full = run_m3(generator, val_records, n_samples=N_SAMPLES,
                         seed_mode="index")
    print(f"M3: test ({N_SAMPLES} samples) ...")
    m3_test_full = run_m3(generator, test_records, n_samples=N_SAMPLES,
                          seed_mode="index")
    m3_val = rescore_m3(m3_val_full, M3_K)
    m3_test = rescore_m3(m3_test_full, M3_K)

    rows, summary = [], {}
    for target in targets:
        # ---- M1 -----------------------------------------------------------
        tau_m1, val_cov1 = tune_threshold(m1_val, target)
        m1_val_t = apply_m1_threshold(m1_val, tau_m1)
        m1_test_t = apply_m1_threshold(m1_test, tau_m1)
        m1m = evaluate_predictions(m1_test_t)
        rows.append(_metric_row("M1", target, tau_m1, val_cov1, m1m))
        save_run(f"m1_test_cov{target}", m1_test_t, m1m)

        # ---- M3 -----------------------------------------------------------
        tau_c, val_cov3 = tune_threshold(m3_val, target)
        m3_val_t = apply_m3_threshold(m3_val, tau_c)
        m3_test_t = apply_m3_threshold(m3_test, tau_c)
        m3m = evaluate_predictions(m3_test_t)
        unreachable = abs(val_cov3 - target) > 0.05
        note = (f"M3 cannot reach {target}: min val coverage {val_cov3:.3f}"
                if unreachable else "")
        rows.append(_metric_row("M3 (k=5)", target, tau_c, val_cov3, m3m, note))
        save_run(f"m3_k5_test_cov{target}", m3_test_t, m3m)

        # ---- two-stage (simulated from the full M1 / M3 predictions) -------
        best = tune_two_stage(m1_val, m3_val, target)
        cfg = best["config"]
        ts_val = simulate_two_stage(m1_val, m3_val, *cfg)
        ts_test = simulate_two_stage(m1_test, m3_test, *cfg)
        tsm = evaluate_predictions(ts_test)
        note = f"tau_low={cfg[0]:.3f}, tau_high={cfg[1]:.3f}, tau_c={cfg[2]:.1f}"
        rows.append(_metric_row("Two-stage", target, cfg[0], best["coverage"], tsm, note))
        save_run(f"twostage_test_cov{target}", ts_test, tsm)

        summary[str(target)] = {"m1": m1m, "m3": m3m, "two_stage": tsm,
                                "two_stage_cfg": list(cfg)}

    table = pd.DataFrame(rows)
    # Keep the curated report table intact; the notebook writes its own copy.
    table.to_csv(os.path.join(OUT_DIR, "notebook_main_table.csv"), index=False)
    with open(os.path.join(OUT_DIR, "notebook_results_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    if run_ablations:
        print("ablation: sample counts {3, 5, 8} ...")
        ablation = pd.DataFrame(sample_count_ablation(generator, val_records, test_records))
        ablation.to_csv(os.path.join(OUT_DIR, "m3_sample_count_ablation.csv"), index=False)

    print("\n=== test table ===")
    display(table)
    return table


def run_quick_demo(generator, records, n=QUICK_N):
    """20-question end-to-end demo for the presentation; thresholds are the
    full-val 0.3 operating point, so this is illustrative, not a new result."""
    generator.load()
    subset = records[:n]
    m1 = run_m1(generator, subset)
    m3 = rescore_m3(run_m3(generator, subset, n_samples=5), 5)
    ts = simulate_two_stage(m1, m3, -0.160, -0.030, 1.0)

    rows = []
    for p in ts:
        rows.append({
            "question": p["question"][:70],
            "m1_conf": round(p["stage1_score"], 3),
            "stage": p["stage"],
            "answer": p["answer"],
            "decision": "ABSTAIN" if p["abstain"] else "ANSWER",
            "correct": p["correct"],
        })
    df = pd.DataFrame(rows)
    display(df)
    print("demo metrics (20 questions, not the report table):")
    print(evaluate_predictions(ts))
    return df


def show_cached_results():
    """Show the report tables without loading the model (fast path for the talk)."""
    for name in ["main_table_fixed_coverage.csv", "m2_m4_table.csv",
                 "ci_table.csv", "pairwise_ci.csv"]:
        path = os.path.join(OUT_DIR, name)
        if os.path.exists(path):
            print("\n== " + name + " ==")
            display(pd.read_csv(path))


RUN_FULL_EXPERIMENT = False
RUN_QUICK_DEMO = True

if RUN_FULL_EXPERIMENT:
    main_table = run_full_pipeline(gen, val_records, test_records)
elif RUN_QUICK_DEMO:
    demo_df = run_quick_demo(gen, val_records, QUICK_N)
else:
    show_cached_results()


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

using slm_interface.SLM (student B's unified interface)


,question,m1_conf,stage,answer,decision,correct
0,How many nations are within the Amazon Basin?,-0.233,low,nine,ABSTAIN,False
1,What welding process was demonstrated in 1901?,-0.085,borderline,oxyacetylene welding,ANSWER,True
2,How is packet switching charecterized,-1.119,low,Packet-switched,ABSTAIN,False
3,How many lakes are there in Warsaw?,-1.151,low,20 lakes,ABSTAIN,False
4,On where are most tourist attractions focused ...,-0.040,borderline,Melbourne,ANSWER,True


demo metrics (20 questions, not the report table):
{'n': 5, 'coverage': 0.4, 'selective_accuracy': 1.0, 'risk': 0.0, 'correct_refusal_rate': nan, 'over_refusal_rate': 0.6, 'correct_answer_refusal_rate': 0.3333333333333333, 'wrong_answer_refusal_rate': 1.0, 'auroc': 1.0, 'auprc': 1.0, 'ece': 0.17011825645506926, 'brier': 0.049979842021690445, 'behaviour_accuracy': 0.4, 'avg_samples': 3.0, 'avg_latency_s': 0.4688913631439209}


## 11. Report mapping and checklist

| Report item | Where it lives |
|---|---|
| Data & Evaluation | sections 1–3 (SQuAD 2.0 parsing, splits, normalisation) and section 5 (metrics) |
| M1 description | section 6 (greedy pass, length-normalised log-prob, `tau_m1` on val) |
| M3 description | section 7 (5 samples, lexical clustering, `tau_c` on val) |
| Two-stage design (Figure 1) | section 8 (three-way split, borderline refinement, cost per question) |
| Threshold sweep / sample-count ablation / cost | `tune_two_stage`, `sample_count_ablation`, `cost_summary` in sections 8–10 |
| Case study | `split_error_cases` and `error_cases_*.json` in section 10 |

Before submission:

- Freeze `test`; every threshold is tuned on `val` only.
- Save the seed, model name, `MAX_NEW_TOKENS`, temperature, `N_SAMPLES` and the chosen thresholds
  with each result file (already written by `run_full_pipeline`).
- If M4 (Part B) slips, the fallback scope is the complete M1 / M3 / two-stage comparison — the
  notebook does not depend on M4.


## 12. Live demo (for the presentation video)

One question end to end: confidence -> route -> decision.
Thresholds are the val-tuned 0.3-coverage operating point
(`results/fixed_coverage_config.json`): `tau_low=-0.160`, `tau_high=-0.030`, `tau_c=1.0`.

The three cases cover: answered correctly / correctly refused / answered although it should refuse.


In [16]:
# ============================================================================
# 12. Live demo - one question end to end (for the presentation video)
# Thresholds: val-tuned operating point for 0.3 coverage
# ============================================================================
TAU_LOW, TAU_HIGH, TAU_C = -0.160, -0.030, 1.0

DEMO_CASES = [
    {
        "label": "answerable",
        "question": "What is the name of the desert city?",
        "context": ("Many locals and tourists frequent the southern California coast for its "
                    "popular beaches, and the desert city of Palm Springs is popular for its "
                    "resort feel and nearby open spaces."),
        "gold": "Palm Springs",
        "is_impossible": False,
    },
    {
        "label": "unanswerable, should refuse",
        "question": "What is the ULCA mascot?",
        "context": ("College sports are also popular in southern California. The UCLA Bruins "
                    "and the USC Trojans both field teams in NCAA Division I in the Pac-12 "
                    "Conference, and there is a longtime rivalry between the schools."),
        "gold": None,
        "is_impossible": True,
    },
    {
        "label": "unanswerable, model answers anyway",
        "question": "Who was pioneer and studied specific complexity measure in 1948?",
        "context": ("Earlier papers studying problems solvable by Turing machines with specific "
                    "bounded resources include John Myhill's definition of linear bounded automata "
                    "(Myhill 1960), Raymond Smullyan's study of rudimentary sets (1961), as well as "
                    "Hisao Yamada's paper on real-time computations (1962). Somewhat earlier, Boris "
                    "Trakhtenbrot (1956), a pioneer in the field from the USSR, studied another "
                    "specific complexity measure."),
        "gold": None,
        "is_impossible": True,
    },
]


def demo(case):
    """Run one question through the two-stage abstainer and print the decision."""
    q, ctx = case["question"], case["context"]
    answers, s1 = gen.generate(q, ctx, n_samples=1, return_logprobs=True, do_sample=False)
    a1 = answers[0]

    if s1 >= TAU_HIGH:
        route, decision, final = "Stage 1: s1 >= tau_high -> answer", "ANSWER", a1
    elif s1 <= TAU_LOW:
        route, decision, final = "Stage 1: s1 <= tau_low -> refuse", "ABSTAIN", "I don't know"
    else:
        extra = gen.generate(q, ctx, n_samples=5, return_logprobs=False, do_sample=True)
        cons, _ = consistency_score([a1] + extra)
        route = "Stage 2: borderline -> 5 samples, consistency=%.1f" % cons
        decision, final = ("ANSWER", a1) if cons >= TAU_C else ("ABSTAIN", "I don't know")

    if decision == "ABSTAIN":
        verdict = "correct refusal" if case["is_impossible"] else "over-refusal"
    elif case["is_impossible"]:
        verdict = "wrong answer (should have refused)"
    else:
        ok = normalize_answer(final) == normalize_answer(case["gold"] or "")
        verdict = "correct" if ok else "incorrect"

    print("=" * 76)
    print("[%s]" % case["label"])
    print("Q: " + q)
    print("Context: " + ctx[:100] + "...")
    print("-" * 76)
    print("confidence (mean token log-prob): %+.3f" % s1)
    print("route    : " + route)
    print("model    : " + final)
    print("decision : " + decision)
    print("verdict  : " + verdict)
    print("=" * 76)
    print()


if getattr(gen, "slm", None) is None and getattr(gen, "model", None) is None:
    gen.load()

for case in DEMO_CASES:
    demo(case)


[answerable]
Q: What is the name of the desert city?
Context: Many locals and tourists frequent the southern California coast for its popular beaches, and the des...
----------------------------------------------------------------------------
confidence (mean token log-prob): -0.016
route    : Stage 1: s1 >= tau_high -> answer
model    : Palm Springs
decision : ANSWER
verdict  : correct

[unanswerable, should refuse]
Q: What is the ULCA mascot?
Context: College sports are also popular in southern California. The UCLA Bruins and the USC Trojans both fie...
----------------------------------------------------------------------------
confidence (mean token log-prob): -1.479
route    : Stage 1: s1 <= tau_low -> refuse
model    : I don't know
decision : ABSTAIN
verdict  : correct refusal

[unanswerable, model answers anyway]
Q: Who was pioneer and studied specific complexity measure in 1948?
Context: Earlier papers studying problems solvable by Turing machines with specific bounded resource